## Prerequisites for .NET compatibility

if you changed c# code, remember to run `dotnet build`, `dotnet publish` and to restart the jupyter kernel (the **last two** are a requisite).

## Imports

In [ ]:
import sys
import os
from pythonnet import load
load("coreclr")


In [ ]:
import clr
import System
sys.path.append(os.path.abspath(r"../../src/HeuristicLib.Extensions/bin/Release/net10.0/publish"))
clr.AddReference("HEAL.HeuristicLib.Extensions")
[a.FullName for a in System.AppDomain.CurrentDomain.GetAssemblies() if "HEAL" in a.FullName]

In [ ]:
from HEAL.HeuristicLib.PythonInterOptScripts import ExtendedSymbolicRegressionProblem
from HEAL.HeuristicLib.Genotypes.Trees import SymbolicExpressionTree
from HEAL.HeuristicLib.Optimization import ObjectiveVector
from System import Func, Array, Double


def score_individual_tree(tree, objective_vector):
  r2 = objective_vector[0] # first supplied objective is R^2 but will be overridden by the combined weighted score
  print(f"individual: R^2: {r2}, Depth: {tree.Depth}, Length: {tree.Length}")
  return [r2 + 0.1 * tree.Depth + 0.1 * tree.Length # weighted sum of R^2 and model interpretability score (ADD YOUR OWN WEIGHTS HERE)
          , r2 # just R^2, so we later compare values
          , tree.Depth  # Dimensional Consistency CHANGE TO ACTUAL IMPLEMENTATION
          , tree.Length  # Limits & Trends CHANGE TO ACTUAL IMPLEMENTATION
          , 0.1 * tree.Depth + 0.1 * tree.Length  # Symmetry CHANGE TO ACTUAL IMPLEMENTATION
  ]

def score_population(trees, objective_vectors):
  results = []
  for tree, objective_vector in zip(trees, objective_vectors):
    r2 = objective_vector[0] # first supplied objective is R^2 but will be overridden by the combined weighted score
    print(f"population: R^2: {r2}, Depth: {tree.Depth}, Length: {tree.Length}")
    results.append([r2 + 0.1 * tree.Depth + 0.1 * tree.Length # weighted sum of R^2 and model interpretability score (ADD YOUR OWN WEIGHTS HERE)
                    , r2 # just R^2, so we later compare values
                    , tree.Depth  # Dimensional Consistency CHANGE TO ACTUAL IMPLEMENTATION
          , tree.Length  # Limits & Trends CHANGE TO ACTUAL IMPLEMENTATION
          , 0.1 * tree.Depth + 0.1 * tree.Length  # Symmetry CHANGE TO ACTUAL IMPLEMENTATION
    ])
  return results

population = ExtendedSymbolicRegressionProblem.RunDefault(
                "../../test/HeuristicLib.Core.Tests/TestData/192_vineyard.tsv",
                50, 
                None, # optional: provide the function down below for scoring individual trees
                # Func[SymbolicExpressionTree, ObjectiveVector, Array[Double]](score_individual_tree),  # optional: pass none if you only want to use the population scoring function alone
                Func[Array[SymbolicExpressionTree], Array[ObjectiveVector], Array[Array[Double]]](score_population) 
              )

## Code

In [ ]:
for solution in population.Solutions:
  print(solution)